**SILVER LAYER**

Data access using service application

In [0]:
# Credentials are stored in the Databricks secret scope 'project1-adls' (never hard-code them here)
storage_account = "azdbstorageaccountt"
client_id     = dbutils.secrets.get(scope="project1-adls", key="sp-client-id")
client_secret = dbutils.secrets.get(scope="project1-adls", key="sp-client-secret")
tenant_id     = dbutils.secrets.get(scope="project1-adls", key="sp-tenant-id")

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

Data loading


_Reading calendar data_

In [0]:

#“Read all CSV data located in the Calendar folder of the raw container in my Azure Storage Account, treat the first row as column names, automatically detect the column data types, and return the result as a Spark DataFrame called df.”

df_Cal=spark.read.format('csv').option("header", True).option("inferSchema",True).load('abfss://raw@azdbstorageaccountt.dfs.core.windows.net/Calendar')

_Reading customers data_

In [0]:
df_Cus=spark.read.format('csv').option("header", True).option("inferSchema",True).load('abfss://raw@azdbstorageaccountt.dfs.core.windows.net/Customers')

_Reading products data_ (the Products folder holds 3 files with different schemas, so each is read separately)

In [0]:
df_Prod=spark.read.format('csv').option("header", True).option("inferSchema",True).load('abfss://raw@azdbstorageaccountt.dfs.core.windows.net/Products/product_lookup.csv')
df_ProdCat=spark.read.format('csv').option("header", True).option("inferSchema",True).load('abfss://raw@azdbstorageaccountt.dfs.core.windows.net/Products/product_categories_lookup.csv')
df_ProdSubCat=spark.read.format('csv').option("header", True).option("inferSchema",True).load('abfss://raw@azdbstorageaccountt.dfs.core.windows.net/Products/product_subcategories_lookup.csv')

_Reading returns data_

In [0]:
df_Ret=spark.read.format('csv').option("header", True).option("inferSchema",True).load('abfss://raw@azdbstorageaccountt.dfs.core.windows.net/Returns')

_Reading sales data_ (2020, 2021 and 2022 files combined)

In [0]:
df_Sales=spark.read.format('csv').option("header", True).option("inferSchema",True).load('abfss://raw@azdbstorageaccountt.dfs.core.windows.net/Sales')

_Reading territories data_

In [0]:
df_Ter=spark.read.format('csv').option("header", True).option("inferSchema",True).load('abfss://raw@azdbstorageaccountt.dfs.core.windows.net/Territories')

In [0]:
# Quick check: row and column counts for every loaded DataFrame
for name, d in {'Calendar': df_Cal, 'Customers': df_Cus, 'Products': df_Prod, 'ProductCategories': df_ProdCat,
                 'ProductSubcategories': df_ProdSubCat, 'Returns': df_Ret, 'Sales': df_Sales, 'Territories': df_Ter}.items():
    print(f"{name}: {d.count()} rows, {len(d.columns)} columns")

Data profiling

_Read-only profiling of every loaded DataFrame (no cleaning yet): table summary, column-level metrics, key checks, datatype issues, referential integrity and business-rule checks_

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, NumericType, DateType, TimestampType

# Registry of loaded DataFrames + their expected business key(s). Profiling is read-only: no DataFrame is modified.
datasets = {
    'Calendar':             (df_Cal,        ['Date']),
    'Customers':            (df_Cus,        ['CustomerKey']),
    'Products':             (df_Prod,       ['ProductKey']),
    'ProductCategories':    (df_ProdCat,    ['ProductCategoryKey']),
    'ProductSubcategories': (df_ProdSubCat, ['ProductSubcategoryKey']),
    'Returns':              (df_Ret,        ['ReturnDate', 'TerritoryKey', 'ProductKey']),
    'Sales':                (df_Sales,      ['OrderNumber', 'OrderLineItem']),
    'Territories':          (df_Ter,        ['SalesTerritoryKey']),
}

def profile_table(name, df, keys):
    """Table-level metrics: rows, columns, schema, duplicate rows, key nulls/duplicates, column-name issues."""
    total = df.count()
    distinct_rows = df.distinct().count()
    key_cols = [F.col(f'`{k}`') for k in keys]
    key_null_rows = df.filter(F.greatest(*[c.isNull().cast('int') for c in key_cols]) == 1).count() if len(keys) > 1 else df.filter(key_cols[0].isNull()).count()
    distinct_keys = df.select(*key_cols).distinct().count()
    bad_col_names = [c for c in df.columns if c != c.strip() or ' ' in c or not c.replace('_', '').isalnum()]
    return {
        'table': name, 'row_count': total, 'column_count': len(df.columns),
        'duplicate_rows': total - distinct_rows,
        'key_columns': ', '.join(keys),
        'key_null_rows': key_null_rows,
        'key_duplicate_rows': total - distinct_keys,
        'key_is_unique': (total == distinct_keys) and key_null_rows == 0,
        'column_name_issues': ', '.join(bad_col_names) if bad_col_names else None,
        'schema': ', '.join(f'{f.name}:{f.dataType.simpleString()}' for f in df.schema.fields),
    }

def profile_columns(name, df):
    """Column-level metrics computed in a single aggregation pass per DataFrame."""
    total = df.count()
    exprs = []
    for f in df.schema.fields:
        c, n, dt = F.col(f'`{f.name}`'), f.name, f.dataType
        exprs += [F.sum(c.isNull().cast('int')).alias(f'{n}__nulls'),
                  F.countDistinct(c).alias(f'{n}__distinct')]
        if isinstance(dt, StringType):
            exprs += [
                F.sum((F.trim(c) == '').cast('int')).alias(f'{n}__empty'),
                F.sum(((c != F.trim(c)) | c.rlike('\\s{2,}')).cast('int')).alias(f'{n}__whitespace'),
                F.countDistinct(F.lower(F.trim(c))).alias(f'{n}__distinct_normalized'),
                F.min(F.length(c)).alias(f'{n}__min'), F.max(F.length(c)).alias(f'{n}__max'),
                # datatype probes: how many non-null values would cast cleanly to a number / date
                F.sum(F.expr(f'try_cast(`{n}` as double)').isNotNull().cast('int')).alias(f'{n}__numeric_castable'),
                F.sum(F.expr(f'try_to_date(`{n}`)').isNotNull().cast('int')).alias(f'{n}__date_castable'),
            ]
        elif isinstance(dt, (NumericType, DateType, TimestampType)):
            exprs += [F.min(c).cast('string').alias(f'{n}__min'), F.max(c).cast('string').alias(f'{n}__max')]
            if isinstance(dt, NumericType):
                exprs += [F.sum((c < 0).cast('int')).alias(f'{n}__negatives'),
                          F.sum((c == 0).cast('int')).alias(f'{n}__zeros')]
            else:
                exprs += [F.sum((c > F.current_date()).cast('int')).alias(f'{n}__future')]
    r = df.agg(*exprs).collect()[0].asDict()

    rows = []
    for f in df.schema.fields:
        n, dt = f.name, f.dataType
        g = lambda k: r.get(f'{n}__{k}')
        nulls, non_null = g('nulls'), total - g('nulls')
        issues = []
        if isinstance(dt, StringType):
            num_ok, date_ok = g('numeric_castable'), g('date_castable')
            if 0 < num_ok < non_null and num_ok / max(non_null, 1) > 0.5:
                issues.append(f'mostly numeric but {non_null - num_ok} non-numeric values')
            elif num_ok == non_null and non_null > 0:
                issues.append('all values numeric -> stored as string')
            if 0 < date_ok == non_null:
                issues.append('all values are dates -> stored as string')
            if g('empty'):
                issues.append(f"{g('empty')} empty/blank strings")
            if g('whitespace'):
                issues.append(f"{g('whitespace')} leading/trailing/double spaces")
            if g('distinct_normalized') < g('distinct'):
                issues.append(f"case/space variants ({g('distinct')} raw vs {g('distinct_normalized')} normalized distinct)")
        if g('negatives'):
            issues.append(f"{g('negatives')} negative values")
        if g('future'):
            issues.append(f"{g('future')} future dates")
        if nulls:
            issues.append(f'{nulls} nulls')
        rows.append({
            'table': name, 'column': n, 'dtype': dt.simpleString(),
            'null_count': nulls, 'null_pct': round(100 * nulls / total, 2) if total else 0.0,
            'distinct_count': g('distinct'), 'distinct_pct': round(100 * g('distinct') / total, 2) if total else 0.0,
            'empty_strings': g('empty'), 'whitespace_issues': g('whitespace'),
            'min_value_or_len': None if g('min') is None else str(g('min')),
            'max_value_or_len': None if g('max') is None else str(g('max')),
            'negatives': g('negatives'), 'zeros': g('zeros'), 'future_dates': g('future'),
            'flags': '; '.join(issues) if issues else None,
        })
    return rows

print('Profiling helpers ready for', len(datasets), 'DataFrames')

_Table-level summary_ (row count, schema, duplicate rows, key uniqueness, column-name issues)

In [0]:
table_schema = ('table string, row_count long, column_count int, duplicate_rows long, key_columns string, '
                'key_null_rows long, key_duplicate_rows long, key_is_unique boolean, column_name_issues string, schema string')
table_profile = spark.createDataFrame([profile_table(n, d, k) for n, (d, k) in datasets.items()], table_schema)
display(table_profile.select('table', 'row_count', 'column_count', 'duplicate_rows', 'key_columns',
                             'key_null_rows', 'key_duplicate_rows', 'key_is_unique', 'column_name_issues', 'schema'))

_Column-level profile_ (nulls, distinct counts, empty strings, whitespace, ranges / string lengths, negatives, future dates, datatype flags)

In [0]:
col_rows = [row for n, (d, _) in datasets.items() for row in profile_columns(n, d)]
col_schema = ('table string, column string, dtype string, null_count long, null_pct double, distinct_count long, '
              'distinct_pct double, empty_strings long, whitespace_issues long, min_value_or_len string, '
              'max_value_or_len string, negatives long, zeros long, future_dates long, flags string')
column_profile = spark.createDataFrame(col_rows, col_schema)
display(column_profile.select('table', 'column', 'dtype', 'null_count', 'null_pct', 'distinct_count', 'distinct_pct',
                              'empty_strings', 'whitespace_issues', 'min_value_or_len', 'max_value_or_len',
                              'negatives', 'zeros', 'future_dates', 'flags'))

_Datatype and value issues_ (offending values behind the flags above, plus value distributions of low-cardinality text columns)

In [0]:
# 1) Values preventing a string column from being numeric (e.g. CustomerKey inferred as string)
for r in column_profile.filter(F.col('flags').contains('non-numeric')).collect():
    d = datasets[r['table']][0]
    print(f"\n{r['table']}.{r['column']} -> non-numeric values:")
    d.filter(F.col(f"`{r['column']}`").isNotNull() & F.expr(f"try_cast(`{r['column']}` as double)").isNull()) \
     .groupBy(f"`{r['column']}`").count().orderBy(F.desc('count')).show(20, truncate=False)

# 2) Distribution of low-cardinality text columns (spot inconsistent codes / casing / unexpected categories)
low_card = column_profile.filter((F.col('dtype') == 'string') & (F.col('distinct_count') <= 15)).collect()
for r in low_card:
    d = datasets[r['table']][0]
    print(f"\n{r['table']}.{r['column']} value distribution:")
    d.groupBy(f"`{r['column']}`").count().orderBy(F.desc('count')).show(20, truncate=False)

_Referential integrity and business-rule checks_ (orphan keys between fact and lookup tables, date coverage, logical rules)

In [0]:
def orphans(child, child_col, parent, parent_col):
    """Count child rows whose key has no match in the parent (compared as trimmed strings to avoid type mismatch)."""
    p = parent.select(F.trim(F.col(parent_col).cast('string')).alias('k')).distinct()
    c = child.select(F.trim(F.col(child_col).cast('string')).alias('k')).filter(F.col('k').isNotNull())
    return c.join(p, 'k', 'left_anti').count()

ri_checks = [
    ('Sales.ProductKey -> Products.ProductKey',                       df_Sales, 'ProductKey', df_Prod, 'ProductKey'),
    ('Sales.CustomerKey -> Customers.CustomerKey',                    df_Sales, 'CustomerKey', df_Cus, 'CustomerKey'),
    ('Sales.TerritoryKey -> Territories.SalesTerritoryKey',           df_Sales, 'TerritoryKey', df_Ter, 'SalesTerritoryKey'),
    ('Sales.OrderDate -> Calendar.Date',                              df_Sales, 'OrderDate', df_Cal, 'Date'),
    ('Returns.ProductKey -> Products.ProductKey',                     df_Ret, 'ProductKey', df_Prod, 'ProductKey'),
    ('Returns.TerritoryKey -> Territories.SalesTerritoryKey',         df_Ret, 'TerritoryKey', df_Ter, 'SalesTerritoryKey'),
    ('Returns.ReturnDate -> Calendar.Date',                           df_Ret, 'ReturnDate', df_Cal, 'Date'),
    ('Products.ProductSubcategoryKey -> Subcategories',               df_Prod, 'ProductSubcategoryKey', df_ProdSubCat, 'ProductSubcategoryKey'),
    ('Subcategories.ProductCategoryKey -> Categories',                df_ProdSubCat, 'ProductCategoryKey', df_ProdCat, 'ProductCategoryKey'),
]
results = [(name, 'referential integrity', orphans(c, cc, p, pc)) for name, c, cc, p, pc in ri_checks]

rule_checks = [
    ('Sales: StockDate after OrderDate',            df_Sales.filter(F.col('StockDate') > F.col('OrderDate'))),
    ('Sales: OrderQuantity <= 0',                   df_Sales.filter(F.col('OrderQuantity') <= 0)),
    ('Returns: ReturnQuantity <= 0',                df_Ret.filter(F.col('ReturnQuantity') <= 0)),
    ('Products: ProductPrice < ProductCost',        df_Prod.filter(F.col('ProductPrice') < F.col('ProductCost'))),
    ('Customers: BirthDate in future / age > 100',  df_Cus.filter((F.col('BirthDate') > F.current_date()) | (F.months_between(F.current_date(), F.col('BirthDate')) / 12 > 100))),
    ('Customers: invalid EmailAddress format',      df_Cus.filter(F.col('EmailAddress').isNotNull() & ~F.col('EmailAddress').rlike('^[^@\\s]+@[^@\\s]+\\.[^@\\s]+$'))),
    ('Customers: AnnualIncome <= 0',                df_Cus.filter(F.col('AnnualIncome') <= 0)),
]
results += [(name, 'business rule', d.count()) for name, d in rule_checks]

# Calendar coverage vs fact dates
cal = df_Cal.agg(F.min('Date').alias('mn'), F.max('Date').alias('mx'), F.count('*').alias('n')).collect()[0]
expected_days = (cal['mx'] - cal['mn']).days + 1
results.append((f"Calendar: missing days between {cal['mn']} and {cal['mx']}", 'coverage', expected_days - df_Cal.select('Date').distinct().count()))

integrity_profile = spark.createDataFrame(results, 'check string, check_type string, violating_rows long')
display(integrity_profile.withColumn('status', F.when(F.col('violating_rows') == 0, 'PASS').otherwise('FAIL')))

# SILVER STAGE: Cleaning and Validation

**Input:** raw DataFrames `df_Cal`, `df_Cus`, `df_Prod`, `df_ProdCat`, `df_ProdSubCat`, `df_Ret`, `df_Sales`, `df_Ter` (loaded above, **never modified**)
**Output:** new `slv_*` DataFrames, `qr_*` quarantine DataFrames, and Silver Delta tables

_Every section has two parts:_
* **PART 1: Real-world knowledge**: definition, the problem it solves, a generic production example and the pattern
* **PART 2: AdventureWorks**: what the profiling says and what we actually apply

_Labels used:_
* **[REQUIRED]** required for this dataset
* **[LEARN]** not required for this dataset, but important to learn
* **[BUSINESS]** depends on business requirements and cannot be assumed

_Every section names the tables it uses in a **Tables used:** line._

## 0. Cross-verify the profiling findings

**Tables used:** `df_Cus` (Customers), `df_Prod` (Products), `df_Sales` (Sales 2020/2021/2022 files), `df_Cal` (Calendar)

_Profiling is already done above. This cell only re-checks the specific rows behind each finding, so the cleaning rules are based on evidence._

In [0]:
from pyspark.sql import functions as F

print('1) Customers rows with a NULL or non-numeric CustomerKey (suspected junk rows):')
df_Cus.filter(F.col('CustomerKey').isNull() | F.expr('try_cast(CustomerKey AS INT)').isNull()).show(truncate=30)

print('2) Customers exact-duplicate rows:')
df_Cus.groupBy(df_Cus.columns).count().filter('count > 1').show(truncate=20)

print('3) Customers duplicated CustomerKey values:')
df_Cus.groupBy('CustomerKey').count().filter('count > 1').show()

print('4) Prefix vs Gender (checks the "m" prefix and NULL prefixes):')
df_Cus.groupBy('Prefix', 'Gender').count().orderBy('Prefix', 'Gender').show()

print('5) BirthDate year distribution (oldest years):')
df_Cus.groupBy(F.year('BirthDate').alias('birth_year')).count().orderBy('birth_year').show(6)

print('6) Email / name casing checks (rows not already lower-case / Proper-case):')
print('   emails not lower-case :', df_Cus.filter('EmailAddress <> lower(EmailAddress)').count())
print('   names not Proper-case :', df_Cus.filter('FirstName <> initcap(FirstName) OR LastName <> initcap(LastName)').count())
df_Cus.filter('FirstName <> initcap(FirstName) OR LastName <> initcap(LastName)').select('FirstName', 'LastName').show(5)

print('7) Products: whitespace in ProductDescription, placeholder ProductColor/ProductStyle')
df_Prod.filter((F.col('ProductDescription') != F.trim('ProductDescription')) | F.col('ProductDescription').rlike('\\s{2,}')) \
       .select(F.concat(F.lit('['), 'ProductDescription', F.lit(']')).alias('desc_with_brackets')).show(3, truncate=80)
df_Prod.groupBy('ProductStyle', 'ProductColor').count().filter("ProductStyle = '0' OR ProductColor = 'NA'").show()

print('8) Sales: does each yearly file have the same schema?')
for y in [2020, 2021, 2022]:
    s = spark.read.format('csv').option('header', True).option('inferSchema', True) \
             .load(f'abfss://raw@azdbstorageaccountt.dfs.core.windows.net/Sales/sales_data_{y}.csv')
    print(f'   sales_data_{y}.csv rows={s.count()} schema={s.schema.simpleString()}')
print('   rows per source file in df_Sales:')
df_Sales.groupBy(F.col('_metadata.file_name').alias('file')).count().orderBy('file').show(truncate=False)

print('9) Calendar columns (are year/month/quarter attributes already there?):', df_Cal.columns)

## 1. Schema and Structure Standardization

### PART 1: Real-World Knowledge

**Definition:** Enforcing consistent column names, correct data types, and stable structure on every table entering the Silver layer.

**Problem it solves:** Raw data arrives with PascalCase, camelCase, spaces, and misdetected types. Without schema contracts, downstream queries break when sources change.

---

**Key patterns a Data Engineer uses:**

**1. Column Renaming → snake\_case** — The analytics standard. Unambiguous, readable, no backtick-quoting in SQL.
*Pattern:* `df.select([F.col(c).alias(to_snake(c)) for c in df.columns])`

**2. Selecting vs Dropping Columns** — `select()` *(whitelist, safer)* vs `drop()` *(blacklist — new source columns leak through)*. **Never drop a column just because you don't understand it.**

**3. Column Reordering** — Keys first → attributes → measures → audit columns. Makes `SELECT *` readable.

**4. Adding Missing Columns** — `df.withColumn('source_system', F.lit('crm'))` when downstream schema contracts require a column the source lacks.

**5. Schema Enforcement** *(explicit schema vs inferSchema)* — `inferSchema` guesses from a sample — one bad row corrupts the type. Production uses `StructType` or DDL strings. *Exactly what happened: `CustomerKey` became string because 5 junk rows aren't numeric.*

**6. Schema Mismatch** — Source adds/removes columns between loads → `unionByName(allowMissingColumns=True)`.

**7. Schema Evolution** — Delta `mergeSchema=True` auto-adds new columns. Must be deliberate. **[LEARN]**

**8. Safe Casting (try\_cast)** — `F.expr("try_cast(col AS INT)")` returns NULL for unparseable values instead of crashing. Use when you expect bad rows and want to flag them.

---

### PART 2: AdventureWorks Implementation

**Tables used:** *All 8 raw DataFrames* — `df_Cal`, `df_Cus`, `df_Prod`, `df_ProdCat`, `df_ProdSubCat`, `df_Ret`, `df_Sales`, `df_Ter`

| Transformation | Label | Justification |
|---|---|---|
| Rename all columns to snake\_case | **[REQUIRED]** | All columns are PascalCase |
| Safe-cast `customer_key` string→int | **[REQUIRED]** | 5 junk rows forced string type |
| Column reordering (keys first) | **[REQUIRED]** | Readability |
| Drop columns | *Not required* | All columns meaningful |
| Add missing columns | *Not required* | No schema gaps |
| Schema enforcement / evolution | **[LEARN]** | Static files, not needed now |

_Creates initial `slv_*` DataFrames. Raw DataFrames **never modified**._

In [0]:
import re
from pyspark.sql import functions as F

# ── Helper: PascalCase → snake_case ──
def to_snake(name):
    s = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1_\2', name)
    s = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', s)
    return s.lower()

def rename_snake(df):
    return df.select([F.col(f'`{c}`').alias(to_snake(c)) for c in df.columns])

# ── Create Silver DataFrames: snake_case + safe casts ──
slv_calendar              = rename_snake(df_Cal)
slv_customers             = rename_snake(df_Cus).withColumn('customer_key', F.expr('try_cast(customer_key AS INT)'))
slv_products              = rename_snake(df_Prod)
slv_product_categories    = rename_snake(df_ProdCat)
slv_product_subcategories = rename_snake(df_ProdSubCat)
slv_returns               = rename_snake(df_Ret)
slv_sales                 = rename_snake(df_Sales)
slv_territories           = rename_snake(df_Ter)

# ── Column reorder: keys first, then attributes ──
slv_customers = slv_customers.select(
    'customer_key','prefix','first_name','last_name','birth_date',
    'marital_status','gender','email_address','annual_income',
    'total_children','education_level','occupation','home_owner')
slv_products = slv_products.select(
    'product_key','product_subcategory_key','product_sku','product_name',
    'model_name','product_description','product_color','product_size',
    'product_style','product_cost','product_price')
slv_sales = slv_sales.select(
    'order_number','order_line_item','order_date','stock_date',
    'product_key','customer_key','territory_key','order_quantity')

# ── Verify schemas ──
for n, d in [('slv_calendar', slv_calendar), ('slv_customers', slv_customers),
             ('slv_products', slv_products), ('slv_product_categories', slv_product_categories),
             ('slv_product_subcategories', slv_product_subcategories), ('slv_returns', slv_returns),
             ('slv_sales', slv_sales), ('slv_territories', slv_territories)]:
    print(f"{n}: {', '.join(f'{f.name}:{f.dataType.simpleString()}' for f in d.schema.fields)}")

print('\n✓ Junk rows now have NULL customer_key after safe cast:')
slv_customers.filter(F.col('customer_key').isNull()).show(truncate=30)

## 2. Date and Timestamp Transformations

### PART 1: Real-World Knowledge

**Definition:** Converting, normalizing, and enriching date/timestamp columns for consistent typing, timezone awareness, and derived time-intelligence attributes.

**Problem it solves:** Dates arrive as strings in dozens of formats (`MM/dd/yyyy`, `dd-Mon-yy`, epoch). Without normalization, date joins fail silently and reports show wrong periods.

---

**Key patterns:**

**1. STRING → DATE/TIMESTAMP** — `F.to_date('col', 'MM/dd/yyyy')`, `F.to_timestamp('col', 'yyyy-MM-dd HH:mm:ss')`. Safe: `F.expr("try_to_date(col, 'format')")`

**2. Timezone Handling** — `F.to_utc_timestamp()` / `F.from_utc_timestamp()`. A timestamp without timezone is ambiguous. **[LEARN]**

**3. Extracting Date Components:**
* `F.year()`, `F.quarter()`, `F.month()` — numeric
* `F.date_format('col', 'MMMM')` — month name; `'EEEE'` — day name
* `F.weekofyear()`, `F.dayofmonth()`, `F.dayofweek()` *(1=Sun … 7=Sat)*

**4. Period Boundaries** — `F.trunc('col','MM')` *(month start)*, `F.last_day()` *(month end)*, `F.trunc('col','quarter')` *(quarter start)*

**5. Date Differences** — `F.datediff()` *(days)*, `F.months_between()` *(fractional months)*

**6. Invalid/Impossible Dates** — Flag `1900-01-01` placeholders, future dates in historical data, `9999-12-31` sentinels.

---

### PART 2: AdventureWorks Implementation

**Tables used:** `slv_calendar` *(Calendar — has only a single `date` column, **no** year/month/quarter attributes)*

| Transformation | Label | Justification |
|---|---|---|
| Extract year, quarter, month, month\_name, day, day\_name, week\_of\_year, day\_of\_week | **[REQUIRED]** | Calendar has only `date` — enrichment needed for time-intelligence |
| Add month\_start, month\_end, quarter\_start, quarter\_end | **[REQUIRED]** | Period-based aggregations in Gold |
| STRING→DATE conversion | *Not required* | All dates already parsed as `date` type |
| Timezone handling | *Not required* | No timestamps in dataset |
| Invalid dates (1900-01-01 in `slv_customers`) | Deferred to **S8** | Junk rows, handled during quarantine |

_Enriches `slv_calendar` with 12 derived date columns._

In [0]:
# ── S2: Enrich slv_calendar with date components ──
# Tables used: slv_calendar

slv_calendar = slv_calendar \
    .withColumn('year',           F.year('date')) \
    .withColumn('quarter',        F.quarter('date')) \
    .withColumn('month',          F.month('date')) \
    .withColumn('month_name',     F.date_format('date', 'MMMM')) \
    .withColumn('week_of_year',   F.weekofyear('date')) \
    .withColumn('day',            F.dayofmonth('date')) \
    .withColumn('day_name',       F.date_format('date', 'EEEE')) \
    .withColumn('day_of_week',    F.dayofweek('date')) \
    .withColumn('month_start',    F.trunc('date', 'MM')) \
    .withColumn('month_end',      F.last_day('date')) \
    .withColumn('quarter_start',  F.trunc('date', 'quarter')) \
    .withColumn('quarter_end',    F.last_day(F.add_months(F.trunc('date', 'quarter'), 2)))

print(f'slv_calendar: {slv_calendar.count()} rows, {len(slv_calendar.columns)} columns')
print(f'Columns: {slv_calendar.columns}')
slv_calendar.orderBy('date').show(5)
slv_calendar.orderBy(F.desc('date')).show(3)

## 3. Text Cleaning and Standardization

### PART 1: Real-World Knowledge

**Definition:** Normalizing string columns so they are consistently cased, free of invisible characters, and contain only meaningful values.

**Problem it solves:** Text data has leading/trailing spaces, double spaces, inconsistent casing (`US` vs `us`), invisible Unicode, and empty strings that look like NULLs but aren't. These cause failed joins, wrong GROUP BY counts, and misleading reports.

---

**Key patterns:**

**1. `trim()`** — Removes leading/trailing whitespace. Always apply to external text. `F.trim('col')`

**2. `upper()` / `lower()` / `initcap()`** — Standardize casing. `initcap` capitalizes first letter of each word → good for names.
> *Caution:* `initcap` produces `Mcdonald` from `MCDONALD`, `Van Der Berg` from `VAN DER BERG`. Only use when data is ALL-CAPS and business accepts the output.

**3. Whitespace Normalization** — Collapse multiple spaces: `F.regexp_replace('col', '\\s+', ' ')`

**4. Empty String → NULL** — `''` is NOT NULL in Spark. Normalize: `F.when(F.trim('col')=='', None).otherwise(F.col('col'))`

**5. `regexp_replace()`** — Remove unwanted chars: `F.regexp_replace('col', '[^a-zA-Z0-9 ]', '')`. Use carefully.

**6. Email Normalization** — `F.lower()` only. Don't alter the local part unnecessarily.

**7. Name Normalization** — `initcap` for `JOHN DOE` → `John Doe`. Only justified when ALL names are caps.

> **Important:** Do NOT clean text just to demonstrate a function. Every transformation must be **justified by profiling evidence**.

---

### PART 2: AdventureWorks Implementation

**Tables used:** `slv_customers` *(Customers)*, `slv_products` *(Products)*

| Transformation | Table | Label | Justification |
|---|---|---|---|
| `initcap()` on `first_name`, `last_name` | `slv_customers` | **[REQUIRED]** | Cross-verify: all 18,148 names are UPPER CASE |
| `trim()` + collapse double spaces in `product_description` | `slv_products` | **[REQUIRED]** | 28 rows have double spaces |
| `lower()` on `email_address` | `slv_customers` | *Not required* | Already lowercase (0 violations) |
| `trim()` on other text columns | *All* | *Not required* | 0 whitespace issues elsewhere |
| Empty string → NULL | *All* | *Not required* | 0 empty strings found |

In [0]:
# ── S3: Text Cleaning ──
# Tables used: slv_customers, slv_products

# Customers: initcap on first_name, last_name (all 18,148 real names are ALL-CAPS)
slv_customers = slv_customers \
    .withColumn('first_name', F.initcap(F.col('first_name'))) \
    .withColumn('last_name',  F.initcap(F.col('last_name')))

# Products: trim + collapse double spaces in product_description (28 rows affected)
slv_products = slv_products \
    .withColumn('product_description', F.trim(F.regexp_replace('product_description', '\\s+', ' ')))

# ── Verify ──
print('✓ slv_customers name casing (BEFORE: ALL-CAPS → AFTER: Initcap):')
slv_customers.filter(F.col('customer_key').isNotNull()).select('first_name','last_name').show(5)

remaining_ws = slv_products.filter(F.col('product_description').rlike('\\s{2,}')).count()
print(f'✓ slv_products: rows with remaining double spaces = {remaining_ws}')
slv_products.select('product_name','product_description').show(3, truncate=80)

## 4. NULL and Missing-Data Handling

### PART 1: Real-World Knowledge

**Definition:** Deciding what to do with NULL/missing values — **per column, per domain** — because there is no single correct approach.

**Problem it solves:** NULL handling done wrong either destroys valid data (dropping rows with optional NULLs) or masks quality issues (replacing everything with 'Unknown').

---

**Strategies (least to most aggressive):**

| Strategy | When to use | Pattern |
|---|---|---|
| **Retain NULL** | Column is optional, NULL is legitimate | Do nothing |
| **Replace with default / 'Unknown'** | Column used in GROUP BY; NULLs create confusing NULL group | `F.coalesce('col', F.lit('Unknown'))` |
| **Imputation** (mean/median/mode) | Numeric column where NULL distorts aggregations | `F.coalesce('col', F.lit(median_val))` |
| **Forward/backward fill** | Time-series with gaps | Window + `F.last(col, ignorenulls=True)` **[LEARN]** |
| **Drop row** | Key column is NULL → row is meaningless | `df.filter(F.col('key').isNotNull())` |
| **Quarantine** | Critical NULLs but shouldn't be silently deleted | Move to `qr_*` table with rejection reason |
| **Flag** | Keep row but mark as suspect | `F.when(F.col('col').isNull(), True).alias('is_col_missing')` |

> **Rule:** NULL handling is **column-specific**. A NULL `prefix` is acceptable; a NULL `customer_key` is fatal. Never apply the same strategy to all columns.

---

### PART 2: AdventureWorks Implementation

**Tables used:** `slv_customers` *(Customers)*, `slv_products` *(Products)*

**NULL Classification:**

| Table.Column | NULLs | Classification | Action |
|---|---|---|---|
| `slv_customers.customer_key` | 6 *(after try\_cast)* | **CRITICAL** | Quarantine in S8 |
| `slv_customers.prefix` | 133 | **OPTIONAL** | Replace with 'Unknown' — used in grouping |
| `slv_customers.first_name` | 6 | **CRITICAL** | Same 6 junk rows → quarantine in S8 |
| `slv_customers.birth_date` | 3 | **OPTIONAL** | Retain NULL — DOB can be unknown |
| `slv_customers.gender` = 'NA' | 130 | **[BUSINESS]** | 'NA' is a placeholder → replace with NULL |
| `slv_customers` other cols | 6 each | **CRITICAL** | Same 6 junk rows → quarantine in S8 |
| `slv_products.product_color` = 'NA' | 50 | **[BUSINESS]** | Placeholder → replace with 'Not Applicable' |

_Key insight: The 6 NULLs across most `slv_customers` columns come from the **same 6 junk rows**. These will be quarantined in S8. The remaining 127 NULL prefixes are legitimate missing data._

_The code below handles **non-critical NULLs only**._

In [0]:
# ── S4: NULL Handling (non-critical only; critical NULLs quarantined in S8) ──
# Tables used: slv_customers, slv_products

# Customers.prefix: 133 NULLs → 'Unknown' for grouping (127 legit missing + 6 junk rows)
slv_customers = slv_customers \
    .withColumn('prefix', F.coalesce(F.col('prefix'), F.lit('Unknown')))

# Customers.gender: 'NA' (130 rows) is a placeholder, not a real gender
# [BUSINESS] decision — treating 'NA' as unknown; replace with NULL
slv_customers = slv_customers \
    .withColumn('gender', F.when(F.upper(F.trim('gender')) == 'NA', None).otherwise(F.col('gender')))

# Products.product_color: 'NA' (50 rows) is a placeholder
# [BUSINESS] decision — replace with 'Not Applicable'
slv_products = slv_products \
    .withColumn('product_color',
                F.when(F.upper(F.trim('product_color')) == 'NA', F.lit('Not Applicable'))
                 .otherwise(F.col('product_color')))

# ── Verify ──
print('✓ slv_customers.prefix after NULL → Unknown:')
slv_customers.groupBy('prefix').count().orderBy(F.desc('count')).show()

print('✓ slv_customers.gender after NA → NULL:')
slv_customers.groupBy('gender').count().orderBy(F.desc('count')).show()

print('✓ slv_products.product_color after NA → Not Applicable:')
slv_products.groupBy('product_color').count().orderBy(F.desc('count')).show()

## 5. Duplicate Handling

### PART 1: Real-World Knowledge

**Definition:** Identifying and removing rows that appear more than once — either as exact copies or as rows sharing the same business key but with differing attributes.

**Problem it solves:** Duplicates inflate counts, double-charge revenue, and break one-to-one joins. They sneak in via re-delivered files, replayed events, union of overlapping partitions, or source-system bugs.

---

**Key patterns:**

**1. Exact Duplicate Detection** — Every column identical.
`df.groupBy(df.columns).count().filter('count > 1')` to detect; `df.dropDuplicates()` to remove.

**2. Key-Based Duplicates** — Same business key, different attribute values (e.g. two addresses for one customer).
`df.groupBy('customer_key').count().filter('count > 1')`

**3. Latest-Record Selection (Window + row\_number)** — Keep the most recent version of each key:

    from pyspark.sql.window import Window
    w = Window.partitionBy('customer_key').orderBy(F.desc('updated_at'))
    df.withColumn('rn', F.row_number().over(w)).filter('rn = 1').drop('rn')

**4. Timestamp-Based Dedup** — When events replay: partition by event ID, keep the earliest (or latest) timestamp. **[LEARN]**

**5. Deterministic Dedup** — When no timestamp exists, define a deterministic rule (e.g. keep the row with the fewest NULLs, or the highest surrogate key).

**6. Duplicate Quarantine** — Don't silently drop — move duplicates to a quarantine table for audit.

> **Critical rule:** Before removing anything, determine the **expected grain/key** of each table. An order can have multiple line items — `OrderNumber` alone does NOT uniquely identify a Sales row; `(OrderNumber, OrderLineItem)` does.

---

### PART 2: AdventureWorks Implementation

**Tables used:** `slv_customers` *(Customers)*

| Table | Expected Key | Exact Dupes | Key Dupes | Action |
|---|---|---|---|---|
| `slv_customers` | `customer_key` | 2 *(the 3× `30---` junk rows)* | 2 *(same rows)* | **[REQUIRED]** `dropDuplicates()` |
| `slv_products` | `product_key` | 0 | 0 | *Not required* |
| `slv_sales` | `(order_number, order_line_item)` | 0 | 0 | *Not required* |
| `slv_returns` | `(return_date, territory_key, product_key)` | 0 | 0 | *Not required* |
| All other tables | respective keys | 0 | 0 | *Not required* |

_Only `slv_customers` has duplicates — and they are junk rows. `dropDuplicates()` reduces the 3 identical `30---` rows to 1, which will then be quarantined in S8._

In [0]:
# ── S5: Duplicate Handling ──
# Tables used: slv_customers

print(f'BEFORE dedup: {slv_customers.count()} rows')

# Remove exact duplicate rows (keeps first occurrence)
slv_customers = slv_customers.dropDuplicates()

print(f'AFTER  dedup: {slv_customers.count()} rows')
print(f'Rows removed: {18154 - slv_customers.count()}')

# Verify: no remaining key duplicates among non-null keys
remaining_key_dupes = slv_customers.filter(F.col('customer_key').isNotNull()) \
    .groupBy('customer_key').count().filter('count > 1').count()
print(f'\n✓ Remaining key duplicates (excluding NULL keys): {remaining_key_dupes}')

## 6. Numeric Cleaning and Validation

### PART 1: Real-World Knowledge

**Definition:** Ensuring numeric columns are correctly typed, within valid ranges, free of formatting artifacts, and contain no impossible values.

**Problem it solves:** Numbers arrive as strings with commas (`"1,234.56"`), currency symbols (`"$99.99"`), trailing whitespace, or as text representations that inferSchema misses. Even when typed correctly, they can contain negative quantities, zero prices, NaN, or Infinity.

---

**Key patterns:**

**1. String → Integer / Decimal** — `F.col('col').cast('int')`, `F.col('col').cast('decimal(10,2)')`. For dirty strings: strip first, then cast.

**2. Comma / Currency Removal** — `F.regexp_replace('price_str', '[\\$,]', '').cast('double')`

**3. Decimal Precision** — `F.round('col', 2)` for financial amounts. Avoid floating-point drift with `DecimalType`.

**4. Negative-Value Validation** — Quantities, prices, ages should never be negative. Flag: `F.when(F.col('qty') < 0, True)`

**5. Zero-Value Validation** — A zero price may be a free item or a data error — context-dependent.

**6. Valid Numeric Ranges** — Domain rules: age 0–120, quantity 1–999, price > 0. Flag outliers.

**7. NaN / Infinity** — `F.isnan('col')`, `df.filter(F.col('col') == float('inf'))`. Replace or quarantine.

**8. Rounding** — `F.round('cost', 4)` for cost; `F.round('price', 2)` for display price.

**9. Units Standardization** — Convert miles ↔ km, Celsius ↔ Fahrenheit when merging international sources. **[LEARN]**

---

### PART 2: AdventureWorks Implementation

**Tables used:** `slv_customers` *(annual\_income, total\_children)*, `slv_products` *(product\_cost, product\_price)*, `slv_sales` *(order\_quantity)*, `slv_returns` *(return\_quantity)*

| Check | Table.Column | Label | Result from Profiling |
|---|---|---|---|
| String → Int cast | `slv_customers.customer_key` | **Already done in S1** | Safe-cast applied |
| Negative values | All numeric cols | *Not required* | 0 negatives found |
| Zero values | `slv_customers.total_children` | *Acceptable* | 5,080 zeros — legitimate (childless customers) |
| Price < Cost | `slv_products` | *Not required* | 0 violations |
| Quantity ≤ 0 | `slv_sales`, `slv_returns` | *Not required* | 0 violations |
| NaN / Infinity | All numeric cols | *Not required* | None found |
| Range validation | `slv_customers.annual_income` | **[REQUIRED]** | Verify 10k–170k is reasonable |

_Most numeric columns are already clean. The code below runs validation checks and confirms no action is needed beyond what S1 already did._

In [0]:
# ── S6: Numeric Cleaning & Validation ──
# Tables used: slv_customers, slv_products, slv_sales, slv_returns

from pyspark.sql.types import NumericType
import math

checks = []
for name, df in [('slv_customers', slv_customers), ('slv_products', slv_products),
                 ('slv_sales', slv_sales), ('slv_returns', slv_returns)]:
    for f in df.schema.fields:
        if isinstance(f.dataType, NumericType):
            stats = df.agg(
                F.min(f.name).alias('min'), F.max(f.name).alias('max'),
                F.sum(F.when(F.col(f.name) < 0, 1).otherwise(0)).alias('negatives'),
                F.sum(F.when(F.isnan(F.col(f.name).cast('double')), 1).otherwise(0)).alias('nans'),
                F.sum(F.when(F.col(f.name) == 0, 1).otherwise(0)).alias('zeros')
            ).collect()[0]
            checks.append((name, f.name, f.dataType.simpleString(),
                           stats['min'], stats['max'], stats['negatives'], stats['nans'], stats['zeros']))

schema = 'table string, column string, dtype string, min_val string, max_val string, negatives long, nans long, zeros long'
df_num_check = spark.createDataFrame(checks, schema)
print('Numeric validation summary (all tables):')
display(df_num_check)

## 7. Categorical Data Standardization

### PART 1: Real-World Knowledge

**Definition:** Ensuring categorical/lookup columns contain only approved, consistently formatted values — no synonyms, no mixed casing, no stale codes.

**Problem it solves:** The same concept appears as `US`, `USA`, `U.S.`, `United States`. Without standardization, GROUP BY produces 4 separate groups instead of 1. Joins on lookup tables fail silently when codes don't match.

---

**Key patterns:**

**1. Inconsistent Casing** — `F.upper()` / `F.lower()` / `F.initcap()` depending on convention.

**2. Abbreviations & Synonyms** — CASE/WHEN mapping:

    F.when(F.col('country').isin('US','USA','U.S.'), 'United States')
     .when(F.col('country').isin('UK','U.K.'), 'United Kingdom')
     .otherwise(F.col('country'))

**3. Mapping Tables** — For large mappings, join to a reference/lookup table instead of hard-coding.

**4. Accepted-Value Validation** — After mapping, check all values are in the approved set: `df.filter(~F.col('col').isin(approved_list))`

**5. Unknown / Other Categories** — Standardize catch-all values: `NULL`, `N/A`, `na`, `Unknown` → single value.

> **Important:** Do NOT assume synonymous values without evidence. `US` and `USA` may be different source-system codes with different meanings. Standardize only when the **business confirms** they mean the same thing.

---

### PART 2: AdventureWorks Implementation

**Tables used:** `slv_customers` *(prefix)*, `slv_products` *(product\_style)*

| Column | Issue | Label | Action |
|---|---|---|---|
| `slv_customers.prefix` = 'm' | Lowercase 'm' (3 rows) | **Junk rows** | These are the `30---` junk rows — will be quarantined in S8; no mapping needed |
| `slv_products.product_style` = '0' | 84 rows with `'0'` alongside `U`, `W`, `M` | **[BUSINESS]** | `'0'` likely means unspecified. Replace with `'U'` (Unisex/Universal) since that's the dominant non-gendered style |
| `slv_territories` | Region/Country/Continent | *Not required* | Values are consistent and complete (verified in profiling) |
| `slv_customers.marital_status` | M/S only | *Not required* | Already clean — only 'M' and 'S' values |
| `slv_customers.home_owner` | Y/N only | *Not required* | Already clean |

_Only `product_style` needs a categorical fix on clean data._

In [0]:
# ── S7: Categorical Data Standardization ──
# Tables used: slv_products

# Products.product_style: '0' (84 rows) → 'U' (Unisex/Universal)
# [BUSINESS] decision — '0' appears to be an unspecified placeholder;
# 'U' is the dominant non-gendered style (174 rows) alongside 'W' (28) and 'M' (7)
slv_products = slv_products \
    .withColumn('product_style',
                F.when(F.col('product_style') == '0', F.lit('U'))
                 .otherwise(F.col('product_style')))

# ── Verify ──
print('✓ slv_products.product_style after 0 → U:')
slv_products.groupBy('product_style').count().orderBy(F.desc('count')).show()

# Also verify: no unexpected categorical values in slv_customers
print('✓ slv_customers.marital_status values (should be M, S only + NULLs from junk):')
slv_customers.groupBy('marital_status').count().orderBy(F.desc('count')).show()

print('✓ slv_customers.home_owner values (should be Y, N only + NULLs from junk):')
slv_customers.groupBy('home_owner').count().orderBy(F.desc('count')).show()

## 8. Filtering and Invalid-Record Handling

### PART 1: Real-World Knowledge

**Definition:** Separating valid records from invalid ones using a **quarantine pattern** so that bad data is never silently dropped and can be investigated.

**Problem it solves:** Silently dropping rows means records vanish without a trace — reconciliation fails, auditors find gaps, and upstream fixes never happen because nobody knows the problem exists.

---

**The production quarantine pattern:**

    Raw DataFrame
         │
    Validation Rules
         ├── valid_df   → continues to Silver
         └── invalid_df → Quarantine table (with rejection_reason)

**Key patterns:**

**1. Row Filters** — `df.filter(F.col('key').isNotNull())` for mandatory fields.

**2. Business-Rule Validation** — `df.filter(F.col('price') > 0)`, `df.filter(F.col('date') <= F.current_date())`

**3. Quality Flags** — Instead of filtering, add a column: `F.when(condition, 'FAIL').otherwise('PASS').alias('dq_status')`

**4. Rejection Reason Column** — `qr_df.withColumn('rejection_reason', F.lit('NULL primary key'))` so the quarantine table is self-documenting.

**5. Quarantine Table** — Write `qr_*` as a Delta table alongside Silver for audit and reprocessing.

---

### PART 2: AdventureWorks Implementation

**Tables used:** `slv_customers` *(Customers — the only table with invalid records)*

**Rows to quarantine (all from `slv_customers`):**

| Row Description | customer\_key | Why Invalid |
|---|---|---|
| `30---` junk row (1 remaining after dedup) | NULL *(try\_cast failed)* | Non-numeric key; all attributes NULL |
| NULL-key row | NULL | Entirely empty row |
| `Export date 20230101 14:59:17` | NULL *(try\_cast failed)* | File export metadata leaked into data |
| `Source AW_Cust_Master` | NULL *(try\_cast failed)* | File source tag leaked into data |

**No other tables have invalid records** — all referential integrity, quantity, and date checks passed in profiling.

_The code below splits `slv_customers` into `slv_customers` (clean) and `qr_customers` (quarantine)._

In [0]:
# ── S8: Filtering & Invalid-Record Handling (Quarantine Pattern) ──
# Tables used: slv_customers

pre_count = slv_customers.count()

# Quarantine: rows with NULL customer_key (failed try_cast in S1 = non-numeric junk)
qr_customers = slv_customers.filter(F.col('customer_key').isNull()) \
    .withColumn('rejection_reason', F.lit('Invalid customer_key: non-numeric or NULL — junk/metadata row'))

# Clean: rows with valid customer_key
slv_customers = slv_customers.filter(F.col('customer_key').isNotNull())

post_count = slv_customers.count()
qr_count = qr_customers.count()

print(f'BEFORE quarantine: {pre_count} rows')
print(f'AFTER  quarantine: {post_count} clean rows + {qr_count} quarantined = {post_count + qr_count} (should equal {pre_count})')
assert pre_count == post_count + qr_count, 'Row count mismatch!'

print(f'\n✓ Quarantined rows ({qr_count}):')
qr_customers.show(truncate=40)

print(f'\n✓ slv_customers sample (clean, {post_count} rows):')
slv_customers.show(5, truncate=30)

## 9. Multiple-File Consolidation

### PART 1: Real-World Knowledge

**Definition:** Combining data that arrives as multiple files (partitioned by date, region, source system) into a single, unified DataFrame or table.

**Problem it solves:** Vendors deliver monthly CSVs, event logs land as hourly Parquet files, legacy systems export per-region. Without consolidation, analysts query N tables instead of 1, miss cross-partition logic, and union bugs creep in.

---

**Key patterns:**

**1. `union()`** — Combines two DataFrames with the **same number and order of columns**. Column names come from the first DF. Fails silently if types differ.

**2. `unionByName()`** — Matches by **column name**, not position. Safer. Use `allowMissingColumns=True` when one file has extra columns.

**3. Folder-Level Read** — Spark can read an entire folder of identically-schemaed files in one call:
`spark.read.csv('path/to/folder/')` — this is what we used for Sales and Calendar.

**4. Schema-Aligned Union** — When files differ slightly, read each, align schemas with `select`, then union:

    df1 = spark.read.csv('2020.csv', header=True, inferSchema=True)
    df2 = spark.read.csv('2021.csv', header=True, inferSchema=True)
    combined = df1.unionByName(df2, allowMissingColumns=True)

**5. Partitioned Historical Data** — Large datasets are partitioned by date/region in storage (`/data/year=2024/month=01/`). Spark reads partition columns automatically.

**6. Incremental File Ingestion** — Auto Loader (`cloudFiles`) incrementally processes new files as they arrive, avoiding full re-reads. **[LEARN]**

---

### PART 2: AdventureWorks Implementation

**Tables used:** `slv_sales` *(Sales — loaded from 3 files: sales\_data\_2020.csv, sales\_data\_2021.csv, sales\_data\_2022.csv)*

| Check | Label | Result |
|---|---|---|
| All 3 Sales files have identical schemas | **Verified in S0** | All `struct<OrderDate:date,...,OrderQuantity:int>` |
| Combined row count matches sum of individual files | **Verified in S0** | 2630 + 23935 + 29481 = 56046 ✓ |
| Spark folder-level read already consolidated them | *Already done* | `df_Sales` was loaded from the Sales/ folder |
| `unionByName` or manual union | *Not required* | Folder read handled it |
| Auto Loader / incremental | **[LEARN]** | Not needed for static files |

_No additional code needed — Sales was already consolidated during raw loading via Spark’s folder-level CSV read. The cross-verify cell confirmed identical schemas and matching row counts._

## 10. Joins and Data Enrichment

### PART 1: Real-World Knowledge

**Definition:** Combining rows from two or more tables based on a related column (join key) to enrich facts with dimensional attributes.

**Problem it solves:** Fact tables (sales, returns) contain only keys; you need dimension tables (customers, products, territories) to answer "who bought what, where." Incorrect joins silently duplicate, drop, or explode rows.

---

**Join types every Data Engineer must know:**

| Type | Keeps | Use Case |
|---|---|---|
| **INNER** | Only matching rows from both sides | Default; when you only want records that exist in both |
| **LEFT** | All left + matching right (NULL if no match) | Fact + dimension; keep all facts even if dimension is missing |
| **RIGHT** | All right + matching left | Rare; usually rewrite as LEFT |
| **FULL OUTER** | All from both sides | Reconciliation, finding mismatches |
| **LEFT SEMI** | Left rows that have a match in right (no right columns) | Filtering: "sales for products that exist" |
| **LEFT ANTI** | Left rows that have **no** match in right | Orphan detection: "sales for products that DON’T exist" |
| **CROSS** | Every combination (cartesian product) | Rare; date-spine generation, unit tests |

**Critical pitfalls:**

* **Duplicate explosion** — A many-to-many join multiplies rows. Always check cardinality: is the relationship 1:1, 1:N, or M:N?
* **Unmatched records** — An INNER join silently drops unmatched rows. Use LEFT + check for NULLs.
* **Select after join** — Always `.select()` only needed columns after a join to avoid ambiguous column names.

**Pattern:**

    enriched = fact_df.join(dim_df, fact_df.product_key == dim_df.product_key, 'left') \
                      .select(fact_df['*'], dim_df['product_name'], dim_df['category'])

---

### PART 2: AdventureWorks Implementation

**Tables used:** *All Silver DataFrames (for planning, not execution)*

**Planned join map for Gold layer (NOT Silver):**

| Fact | Dimension | Join Key | Type |
|---|---|---|---|
| `slv_sales` | `slv_customers` | `customer_key` | LEFT |
| `slv_sales` | `slv_products` | `product_key` | LEFT |
| `slv_sales` | `slv_territories` | `territory_key` = `sales_territory_key` | LEFT |
| `slv_sales` | `slv_calendar` | `order_date` = `date` | LEFT |
| `slv_returns` | `slv_products` | `product_key` | LEFT |
| `slv_returns` | `slv_territories` | `territory_key` = `sales_territory_key` | LEFT |
| `slv_products` | `slv_product_subcategories` | `product_subcategory_key` | LEFT |
| `slv_product_subcategories` | `slv_product_categories` | `product_category_key` | LEFT |

> **Why not join in Silver?** Silver is the **cleaned, validated, single-entity** layer. Joins that combine entities belong in **Gold** where business-ready star schemas and aggregates are built. Joining in Silver creates tightly coupled tables that are harder to maintain and debug.

_No join code in this section — joins will be implemented in the Gold layer notebook._

## 11. Referential Integrity Validation

### PART 1: Real-World Knowledge

**Definition:** Verifying that every foreign key in a child/fact table has a matching primary key in the parent/dimension table.

**Problem it solves:** Orphan records (sales for a product that doesn’t exist, returns for an unknown territory) produce NULL dimensions after joins, break aggregations, and indicate upstream data issues.

---

**Key patterns:**

**1. Primary/Business Key Validation** — `df.groupBy('pk').count().filter('count > 1')` — the key must be unique and non-null.

**2. Foreign-Key Validation (LEFT ANTI join)** — Find orphans:

    orphans = child.join(parent, child.fk == parent.pk, 'left_anti')
    print(f"Orphan rows: {orphans.count()}")

**3. Missing-Reference Quarantine** — In production, orphan rows are moved to a quarantine table rather than silently dropped.

**4. Duplicate-Key Detection** — Before trusting a dimension, verify its PK is unique. A duplicate dimension key causes row explosion on join.

---

### PART 2: AdventureWorks Implementation

**Tables used:** `slv_sales`, `slv_returns`, `slv_customers`, `slv_products`, `slv_product_subcategories`, `slv_product_categories`, `slv_territories`, `slv_calendar`

_Re-running referential integrity checks on the **cleaned Silver DataFrames** (after S1–S8 transformations) to confirm no orphans were introduced by cleaning._

In [0]:
# ── S11: Referential Integrity Validation on Clean Silver DataFrames ──
# Tables used: all slv_* DataFrames

def check_ri(child, child_col, parent, parent_col, label):
    orphan_count = child.join(parent, child[child_col] == parent[parent_col], 'left_anti').count()
    status = 'PASS' if orphan_count == 0 else 'FAIL'
    return (label, orphan_count, status)

ri_results = [
    check_ri(slv_sales, 'product_key', slv_products, 'product_key',
             'slv_sales.product_key → slv_products.product_key'),
    check_ri(slv_sales, 'customer_key', slv_customers, 'customer_key',
             'slv_sales.customer_key → slv_customers.customer_key'),
    check_ri(slv_sales, 'territory_key', slv_territories, 'sales_territory_key',
             'slv_sales.territory_key → slv_territories.sales_territory_key'),
    check_ri(slv_sales, 'order_date', slv_calendar, 'date',
             'slv_sales.order_date → slv_calendar.date'),
    check_ri(slv_returns, 'product_key', slv_products, 'product_key',
             'slv_returns.product_key → slv_products.product_key'),
    check_ri(slv_returns, 'territory_key', slv_territories, 'sales_territory_key',
             'slv_returns.territory_key → slv_territories.sales_territory_key'),
    check_ri(slv_returns, 'return_date', slv_calendar, 'date',
             'slv_returns.return_date → slv_calendar.date'),
    check_ri(slv_products, 'product_subcategory_key', slv_product_subcategories, 'product_subcategory_key',
             'slv_products.product_subcategory_key → slv_product_subcategories'),
    check_ri(slv_product_subcategories, 'product_category_key', slv_product_categories, 'product_category_key',
             'slv_product_subcategories.product_category_key → slv_product_categories'),
]

ri_df = spark.createDataFrame(ri_results, 'check string, orphan_rows long, status string')
display(ri_df)

## 12. Derived Columns

### PART 1: Real-World Knowledge

**Definition:** Adding new columns computed from existing data — either for **technical/audit purposes** or to pre-compute commonly needed attributes.

**Problem it solves:** Without audit columns, you can’t trace when a row was loaded or from which file. Without pre-computed attributes like `full_name` or `age`, every downstream query recomputes them.

---

**Categories of derived columns:**

**TECHNICAL / AUDIT** *(belong in Silver):*
* `ingestion_timestamp` — when the row was loaded: `F.current_timestamp()`
* `source_file` — which file the row came from: `F.input_file_name()`
* `source_system` — hard-coded origin: `F.lit('adventure_works')`
* `batch_id / run_id` — for pipeline traceability

**DATA-QUALITY FLAGS** *(belong in Silver):*
* `is_valid` — `F.lit(True)` for clean rows (quarantined rows got `False`)
* `has_null_gender` — flags rows where a [BUSINESS] decision was made

**GENERAL DERIVED** *(Silver-appropriate):*
* `full_name` — `F.concat_ws(' ', 'first_name', 'last_name')`
* `age` — `F.floor(F.months_between(F.current_date(), 'birth_date') / 12)`

**BUSINESS KPIs** *(belong in Gold, NOT Silver):*
* `revenue` = quantity × price — requires a join between Sales and Products
* `profit_margin` = (price − cost) / price
* `customer_lifetime_value` — aggregation across all orders

> **Rule:** Silver derived columns should be **row-level, single-table** computations. Anything requiring joins or aggregations belongs in Gold.

---

### PART 2: AdventureWorks Implementation

**Tables used:** `slv_customers` *(full\_name, age)*, all `slv_*` *(audit columns)*

| Derived Column | Table | Label | Justification |
|---|---|---|---|
| `full_name` | `slv_customers` | **[REQUIRED]** | Commonly needed; avoids repeated concat |
| `age` | `slv_customers` | **[REQUIRED]** | Useful for segmentation; single-table derivation |
| `ingestion_ts` | All `slv_*` | **[REQUIRED]** | Audit trail — when was this Silver layer built |
| `source_system` | All `slv_*` | **[REQUIRED]** | Identifies origin; critical when Silver merges multiple sources |
| `revenue`, `margin` | *Not here* | **Gold layer** | Requires Sales × Products join |

In [0]:
# ── S12: Derived Columns ──
# Tables used: slv_customers, all slv_* DataFrames

# --- Customer-specific derived columns ---
slv_customers = slv_customers \
    .withColumn('full_name', F.concat_ws(' ', 'first_name', 'last_name')) \
    .withColumn('age', F.floor(F.months_between(F.current_date(), F.col('birth_date')) / 12).cast('int'))

# --- Audit columns on ALL Silver DataFrames ---
from datetime import datetime
_ingest_ts = F.current_timestamp()
_source = F.lit('adventure_works')

slv_calendar              = slv_calendar.withColumn('ingestion_ts', _ingest_ts).withColumn('source_system', _source)
slv_customers             = slv_customers.withColumn('ingestion_ts', _ingest_ts).withColumn('source_system', _source)
slv_products              = slv_products.withColumn('ingestion_ts', _ingest_ts).withColumn('source_system', _source)
slv_product_categories    = slv_product_categories.withColumn('ingestion_ts', _ingest_ts).withColumn('source_system', _source)
slv_product_subcategories = slv_product_subcategories.withColumn('ingestion_ts', _ingest_ts).withColumn('source_system', _source)
slv_returns               = slv_returns.withColumn('ingestion_ts', _ingest_ts).withColumn('source_system', _source)
slv_sales                 = slv_sales.withColumn('ingestion_ts', _ingest_ts).withColumn('source_system', _source)
slv_territories           = slv_territories.withColumn('ingestion_ts', _ingest_ts).withColumn('source_system', _source)

# --- Verify ---
print('✓ slv_customers derived columns (full_name, age, audit):')
slv_customers.select('customer_key','full_name','age','birth_date','ingestion_ts','source_system').show(5, truncate=30)

print('✓ slv_sales audit columns:')
slv_sales.select('order_number','order_date','ingestion_ts','source_system').show(3, truncate=30)

## 13. Hashing and Change Detection

### PART 1: Real-World Knowledge

**Definition:** Computing a deterministic hash of selected columns so that two versions of the same row can be compared **without checking every column individually**.

**Problem it solves:** In incremental pipelines you receive the full current state of a dimension (e.g. a nightly customer export). You need to detect which rows are new (INSERT), which changed (UPDATE), and which are unchanged (SKIP). Comparing 20 columns one by one is expensive; comparing a single hash is O(1).

---

**Key patterns:**

**1. Row Hash** — Concatenate all non-key attribute columns, then hash:

    df.withColumn('row_hash',
        F.sha2(F.concat_ws('||', 'first_name','last_name','email','income'), 256))

**2. `concat_ws()`** — Joins columns with a separator. Use a delimiter unlikely to appear in data (`||`, `\x01`).

**3. `sha2(256)`** — SHA-256 is collision-resistant and deterministic. `md5()` is faster but has known collision weaknesses — acceptable for change detection, not for security.

**4. Comparison Pattern:**

    # incoming vs existing
    incoming.alias('i').join(existing.alias('e'), 'customer_key', 'full') \
        .withColumn('action',
            F.when(F.col('e.row_hash').isNull(), 'INSERT')
             .when(F.col('i.row_hash') != F.col('e.row_hash'), 'UPDATE')
             .otherwise('SKIP'))

**5. Business Key vs Surrogate Key** — The hash covers **attribute columns only**, not the business key (which is the join key) and not audit columns (which change every load).

> **[LEARN]** This pattern becomes essential with incremental pipelines, SCD Type 2, and Delta MERGE. For AdventureWorks’ one-time full load, hashing is not operationally required, but we add it to learn the pattern.

---

### PART 2: AdventureWorks Implementation

**Tables used:** `slv_customers` *(demonstrating the pattern)*, `slv_products`

| Transformation | Label | Justification |
|---|---|---|
| Add `row_hash` on `slv_customers` | **[LEARN]** | Useful for future incremental loads; not needed for initial full load |
| Add `row_hash` on `slv_products` | **[LEARN]** | Same rationale |
| Skip on fact tables | *Not required* | Facts are append-only; change detection is for dimensions |

_The code adds a `row_hash` column to dimension tables, hashing only attribute columns (excluding keys and audit columns)._

In [0]:
# ── S13: Hashing & Change Detection [LEARN] ──
# Tables used: slv_customers, slv_products

# Customer row hash: all attribute columns EXCEPT key, audit, and derived columns
cust_hash_cols = ['prefix','first_name','last_name','birth_date','marital_status',
                  'gender','email_address','annual_income','total_children',
                  'education_level','occupation','home_owner']
slv_customers = slv_customers.withColumn(
    'row_hash', F.sha2(F.concat_ws('||', *[F.coalesce(F.col(c).cast('string'), F.lit('__NULL__')) for c in cust_hash_cols]), 256))

# Product row hash
prod_hash_cols = ['product_subcategory_key','product_sku','product_name','model_name',
                  'product_description','product_color','product_size','product_style',
                  'product_cost','product_price']
slv_products = slv_products.withColumn(
    'row_hash', F.sha2(F.concat_ws('||', *[F.coalesce(F.col(c).cast('string'), F.lit('__NULL__')) for c in prod_hash_cols]), 256))

print('✓ slv_customers with row_hash:')
slv_customers.select('customer_key','full_name','row_hash').show(3, truncate=40)
print('✓ slv_products with row_hash:')
slv_products.select('product_key','product_name','row_hash').show(3, truncate=40)

## 14. Incremental Processing

### PART 1: Real-World Knowledge

**Definition:** Processing **only new or changed data** instead of re-reading the entire dataset every run.

**Problem it solves:** Full reloads are simple but expensive. A 100-million-row table that grows by 50K rows/day should not re-process all 100M every night. Incremental processing reduces compute cost, runtime, and downstream churn.

---

**Loading strategies (least to most sophisticated):**

| Strategy | How it works | When to use |
|---|---|---|
| **Full Load (overwrite)** | Drop + reload entire table every run | Small tables, reference data, first load |
| **Append** | Add new rows only; never touch existing | Event/fact tables with no updates |
| **Overwrite partition** | Replace one date/region partition | Partitioned fact tables with late-arriving data |
| **Incremental (high-water-mark)** | Track the max `updated_at` or `id` from last run; next run reads only rows above that mark | Dimensions with an audit timestamp |
| **Upsert (MERGE)** | INSERT new + UPDATE changed + optionally DELETE removed | Slowly changing dimensions |
| **CDC (Change Data Capture)** | Source publishes INSERT/UPDATE/DELETE events; consumer applies them in order | Real-time or near-real-time replication |
| **Watermark (Structured Streaming)** | Spark tracks event-time progress and drops late data beyond a threshold | Streaming fact ingestion |

**High-water-mark pattern:**

    last_loaded = spark.sql("SELECT max(updated_at) FROM silver.customers").collect()[0][0]
    new_rows = source_df.filter(F.col('updated_at') > last_loaded)
    # → MERGE new_rows into silver.customers

**Auto Loader** (Databricks) — `cloudFiles` format tracks which files have been processed; new files are picked up automatically. **[LEARN]**

---

### PART 2: AdventureWorks Implementation

**Tables used:** *None (teaching only)*

| Strategy | Label | Justification |
|---|---|---|
| Full Load (overwrite) | **Used for this project** | First-time load of static CSV files |
| Append | *Not required now* | No new files arriving |
| Incremental / MERGE | **[LEARN]** | Critical for production; demonstrated in S15 |
| CDC | **[LEARN]** | Requires source-system support (e.g. Debezium, Fivetran) |
| Auto Loader | **[LEARN]** | For file-based incremental ingestion on Databricks |

_No code in this section — AdventureWorks is a one-time full load. The MERGE pattern is demonstrated in S15._

## 15. Delta MERGE (Upsert)

### PART 1: Real-World Knowledge

**Definition:** A single atomic operation that compares incoming data against an existing Delta table and performs INSERT, UPDATE, and optionally DELETE in one pass.

**Problem it solves:** Without MERGE, you need separate INSERT and UPDATE statements, which creates a window where the table is inconsistent. MERGE is atomic — readers see either the old state or the new state, never a mix.

---

**The production MERGE pattern:**

    from delta.tables import DeltaTable

    target = DeltaTable.forName(spark, 'silver.customers')

    target.alias('t').merge(
        incoming_df.alias('s'),
        't.customer_key = s.customer_key'
    ).whenMatchedUpdate(
        condition = 't.row_hash != s.row_hash',   # only update if something changed
        set = { 'first_name': 's.first_name', ... , 'row_hash': 's.row_hash' }
    ).whenNotMatchedInsertAll(
    ).execute()

**Key concepts:**

* **WHEN MATCHED UPDATE** — Row exists in both; update attributes. Add `condition` to skip unchanged rows (using `row_hash` from S13).
* **WHEN NOT MATCHED INSERT** — Row is new; insert it.
* **WHEN NOT MATCHED BY SOURCE DELETE** — Row exists in target but not in source; optionally soft-delete or hard-delete. Use with caution.
* **Avoiding duplicate inserts** — Always match on the **business key**, not on a hash or surrogate.
* **`insertAll()` / `updateAll()`** — Shorthand when source and target schemas match.

---

### PART 2: AdventureWorks Implementation

**Tables used:** *None (teaching only — first load uses overwrite, not MERGE)*

| Scenario | Label | Justification |
|---|---|---|
| First load | **overwrite** | No existing Delta table to merge into |
| Subsequent loads | **MERGE [LEARN]** | Would use the pattern above with `row_hash` for change detection |

_No MERGE code is executed for this initial load. When the pipeline is run a second time (e.g. with updated CSVs), MERGE + row\_hash from S13 would detect only changed customers and update them without re-writing the entire table._

## 16. Data Quality Checks

### PART 1: Real-World Knowledge

**Definition:** Automated checks that run **after transformation** to verify the Silver layer meets its quality contract before downstream consumers (Gold, dashboards, ML) read it.

**Problem it solves:** A pipeline that runs without errors can still produce bad data (wrong counts, broken keys, stale dates). Quality checks catch these before they propagate.

---

**The eight quality dimensions:**

| Dimension | What it checks | Example |
|---|---|---|
| **Completeness** | No unexpected NULLs in required columns | `customer_key IS NOT NULL` |
| **Uniqueness** | Primary keys have no duplicates | `COUNT(DISTINCT pk) = COUNT(*)` |
| **Validity** | Values fall within allowed set/range | `gender IN ('M','F') OR gender IS NULL` |
| **Consistency** | Cross-column rules hold | `stock_date <= order_date` (never stocked after ordered) |
| **Accuracy** | Values match a reference truth | *Rare at Silver — usually Gold/reporting* |
| **Referential Integrity** | FKs match PKs | Already checked in S11 |
| **Freshness** | Data is not stale | `MAX(order_date) >= '2022-06-01'` |
| **Volume** | Row count is within expected range | `count BETWEEN 50000 AND 60000` |

**Production pattern:**

    checks = [('completeness', df.filter('pk IS NULL').count() == 0),
              ('uniqueness',   df.select('pk').distinct().count() == df.count()), ...]
    for name, passed in checks:
        print(f'{name}: {"PASS" if passed else "FAIL"}')

---

### PART 2: AdventureWorks Implementation

**Tables used:** All `slv_*` DataFrames *(post-transformation)*

_Running automated quality checks on the final Silver DataFrames._

In [0]:
# ── S16: Data Quality Checks on Final Silver DataFrames ──
# Tables used: all slv_* DataFrames

dq_results = []

def dq(table, dimension, check_name, passed):
    dq_results.append((table, dimension, check_name, 'PASS' if passed else 'FAIL'))

# --- slv_customers ---
c = slv_customers
c_cnt = c.count()
dq('slv_customers', 'completeness', 'customer_key NOT NULL',   c.filter('customer_key IS NULL').count() == 0)
dq('slv_customers', 'completeness', 'email_address NOT NULL',  c.filter('email_address IS NULL').count() == 0)
dq('slv_customers', 'uniqueness',   'customer_key unique',     c.select('customer_key').distinct().count() == c_cnt)
dq('slv_customers', 'validity',     'gender IN (M,F,NULL)',    c.filter("gender IS NOT NULL AND gender NOT IN ('M','F')").count() == 0)
dq('slv_customers', 'validity',     'annual_income > 0',       c.filter('annual_income <= 0').count() == 0)
dq('slv_customers', 'volume',       'count 18000-19000',       18000 <= c_cnt <= 19000)

# --- slv_products ---
p = slv_products
p_cnt = p.count()
dq('slv_products', 'completeness', 'product_key NOT NULL',     p.filter('product_key IS NULL').count() == 0)
dq('slv_products', 'uniqueness',   'product_key unique',       p.select('product_key').distinct().count() == p_cnt)
dq('slv_products', 'validity',     'price >= cost',            p.filter('product_price < product_cost').count() == 0)

# --- slv_sales ---
s = slv_sales
s_cnt = s.count()
dq('slv_sales', 'completeness', 'no NULLs in key cols',       s.filter('order_number IS NULL OR product_key IS NULL OR customer_key IS NULL').count() == 0)
dq('slv_sales', 'uniqueness',   '(order_number,line_item) unique', s.select('order_number','order_line_item').distinct().count() == s_cnt)
dq('slv_sales', 'validity',     'order_quantity > 0',          s.filter('order_quantity <= 0').count() == 0)
dq('slv_sales', 'consistency',  'stock_date <= order_date',    s.filter('stock_date > order_date').count() == 0)
dq('slv_sales', 'freshness',    'max order_date >= 2022-06',   s.agg(F.max('order_date')).collect()[0][0].strftime('%Y-%m') >= '2022-06')
dq('slv_sales', 'volume',       'count 55000-57000',           55000 <= s_cnt <= 57000)

# --- slv_returns ---
r = slv_returns
dq('slv_returns', 'completeness', 'no NULLs in key cols',     r.filter('return_date IS NULL OR product_key IS NULL').count() == 0)
dq('slv_returns', 'validity',     'return_quantity > 0',       r.filter('return_quantity <= 0').count() == 0)

# --- slv_calendar ---
dq('slv_calendar', 'uniqueness',  'date unique',              slv_calendar.select('date').distinct().count() == slv_calendar.count())
dq('slv_calendar', 'volume',      'count == 912',             slv_calendar.count() == 912)

# --- Display results ---
dq_df = spark.createDataFrame(dq_results, 'table string, dimension string, check_name string, status string')
print(f'Total checks: {dq_df.count()}, Passed: {dq_df.filter("status=\'PASS\'").count()}, Failed: {dq_df.filter("status=\'FAIL\'").count()}')
display(dq_df)

## 17. Silver Reconciliation

### PART 1: Real-World Knowledge

**Definition:** A row-count reconciliation report that proves every raw record is accounted for — either in Silver, in Quarantine, or removed as a documented duplicate.

**Problem it solves:** Without reconciliation, rows silently disappear during transformation. Auditors, regulators, and data consumers need proof that `raw = silver + quarantine + deduped`.

---

**Pattern:**

    assert raw_count == silver_count + quarantine_count + dedup_removed

---

### PART 2: AdventureWorks Implementation

**Tables used:** All raw `df_*` and all `slv_*` DataFrames, `qr_customers`

_The code below builds a reconciliation report for every dataset._

In [0]:
# ── S17: Silver Reconciliation ──
# Tables used: all raw df_* and slv_* DataFrames, qr_customers

recon = []

# Customers: raw → dedup → quarantine → silver
raw_cus = df_Cus.count()          # 18154
slv_cus = slv_customers.count()   # 18148
qr_cus  = qr_customers.count()    # 2
dedup_cus = raw_cus - slv_cus - qr_cus  # 4
recon.append(('Customers', raw_cus, dedup_cus, qr_cus, slv_cus, raw_cus == slv_cus + qr_cus + dedup_cus))

# All other tables: no dedup, no quarantine
for name, raw_df, slv_df in [
    ('Calendar',             df_Cal,        slv_calendar),
    ('Products',             df_Prod,       slv_products),
    ('ProductCategories',    df_ProdCat,    slv_product_categories),
    ('ProductSubcategories', df_ProdSubCat, slv_product_subcategories),
    ('Returns',              df_Ret,        slv_returns),
    ('Sales',                df_Sales,      slv_sales),
    ('Territories',          df_Ter,        slv_territories),
]:
    rc, sc = raw_df.count(), slv_df.count()
    recon.append((name, rc, 0, 0, sc, rc == sc))

recon_df = spark.createDataFrame(recon, 'table string, raw_rows long, dedup_removed long, quarantined long, silver_rows long, reconciled boolean')
print('Silver Reconciliation Report:')
display(recon_df)

# Assert all reconcile
failed = [r['table'] for r in recon_df.filter('reconciled = false').collect()]
assert len(failed) == 0, f'Reconciliation FAILED for: {failed}'
print('\n✓ ALL TABLES RECONCILED SUCCESSFULLY')

## 18. Silver Delta Output

### PART 1: Real-World Knowledge

**Definition:** Writing the validated, transformed Silver DataFrames as **Delta tables** — the standard format for Lakehouse analytics.

**Problem it solves:** DataFrames are ephemeral (lost when the cluster restarts). Delta tables are persistent, versioned, ACID-compliant, and queryable by SQL, BI tools, and ML pipelines.

---

**Key decisions before writing:**

| Decision | Options | Our Choice |
|---|---|---|
| **Format** | Delta *(default on Databricks)* | Delta |
| **Managed vs External** | Managed = Unity Catalog controls storage; External = you specify path | External (ADLS `silver` container) for this project |
| **Write mode** | `overwrite` (full replace), `append`, `merge` | `overwrite` for first load |
| **Partitioning** | By date, region, etc. | Not needed for these small tables |

**Why Delta?**
* ACID transactions — readers never see partial writes
* Time travel — query previous versions with `VERSION AS OF`
* Schema enforcement — rejects writes that don’t match the schema
* Optimized reads — Z-ordering, data skipping, file compaction

**Pattern:**

    df.write.format('delta').mode('overwrite').save('abfss://silver@storage/table_name')
    # or managed: df.write.saveAsTable('catalog.schema.table_name')

---

### PART 2: AdventureWorks Implementation

**Tables written to:** `abfss://silver@azdbstorageaccountt.dfs.core.windows.net/`

| Silver Table | Source | Rows | Key | Mode |
|---|---|---|---|---|
| `silver_calendar` | `slv_calendar` | 912 | `date` | overwrite |
| `silver_customers` | `slv_customers` | 18,148 | `customer_key` | overwrite |
| `silver_products` | `slv_products` | 293 | `product_key` | overwrite |
| `silver_product_categories` | `slv_product_categories` | 4 | `product_category_key` | overwrite |
| `silver_product_subcategories` | `slv_product_subcategories` | 37 | `product_subcategory_key` | overwrite |
| `silver_returns` | `slv_returns` | 1,809 | `(return_date, territory_key, product_key)` | overwrite |
| `silver_sales` | `slv_sales` | 56,046 | `(order_number, order_line_item)` | overwrite |
| `silver_territories` | `slv_territories` | 10 | `sales_territory_key` | overwrite |
| `quarantine_customers` | `qr_customers` | 2 | — | overwrite |

_First-time full load — using `overwrite` mode. Subsequent loads would use MERGE (S15 pattern)._

In [0]:
# ── S18: Write Silver Delta Tables ──
# Target: abfss://silver@azdbstorageaccountt.dfs.core.windows.net/

silver_base = 'abfss://silver@azdbstorageaccountt.dfs.core.windows.net'

tables_to_write = [
    ('silver_calendar',              slv_calendar),
    ('silver_customers',             slv_customers),
    ('silver_products',              slv_products),
    ('silver_product_categories',    slv_product_categories),
    ('silver_product_subcategories', slv_product_subcategories),
    ('silver_returns',               slv_returns),
    ('silver_sales',                 slv_sales),
    ('silver_territories',           slv_territories),
    ('quarantine_customers',         qr_customers),
]

for name, df in tables_to_write:
    path = f'{silver_base}/{name}'
    df.write.format('delta').mode('overwrite').save(path)
    written = spark.read.format('delta').load(path).count()
    print(f'✓ {name}: wrote {written} rows to {path}')

print('\n── All Silver Delta tables written successfully ──')